# FailureGPT — Material Failure Analysis
Run the cells below in order. A public link will appear at the end.

You will need:
- An [Anthropic API key](https://console.anthropic.com) (starts with `sk-ant-`)
- A free [ngrok account](https://dashboard.ngrok.com/signup) and auth token

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
!pip install flask anthropic pyngrok -q

In [ ]:
# ── Cell 2: Download UI from GitHub ───────────────────────────────────────────
# Replace with your own repo URL after forking
GITHUB_RAW_URL = "https://raw.githubusercontent.com/YOUR_USERNAME/failuregpt/main/failuregpt.html"

!wget -q -O failuregpt.html "{GITHUB_RAW_URL}"
print("Downloaded failuregpt.html ✓")

In [ ]:
# ── Cell 3: Configure keys ────────────────────────────────────────────────────
ANTHROPIC_API_KEY = "sk-ant-..."   # paste your Anthropic key here
NGROK_AUTH_TOKEN  = "..."          # paste your ngrok auth token here

In [ ]:
# ── Cell 4: Launch server ─────────────────────────────────────────────────────
from flask import Flask, request, jsonify
from pyngrok import ngrok, conf
import anthropic, threading

conf.get_default().auth_token = NGROK_AUTH_TOKEN
client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
app = Flask(__name__)

with open("failuregpt.html") as f:
    HTML = f.read()

@app.route("/")
def index():
    return HTML

@app.route("/chat", methods=["POST"])
def chat():
    body = request.json
    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=1024,
        system=body["system"],
        messages=body["messages"]
    )
    return jsonify({"content": [{"text": b.text} for b in response.content]})

public_url = ngrok.connect(5000)
print(f"\n  Open FailureGPT here → {public_url}\n")

threading.Thread(target=lambda: app.run(port=5000, use_reloader=False)).start()